# Seminar 01 — Serving-system baseline

**Topic:** Modern LLM inference systems & production challenges  
**Deliverable:** a pinned, machine-readable single-request baseline plus a submission repository that passes the round-01 self-check.

This notebook deliberately starts with `transformers`, not a production server. It gives us a controlled reference for model loading, token counts, cold/warm timing and streaming. Later topics will replace the execution layer while preserving the workload contract.

## 0. Contract before code

For the required run, keep these constants unchanged. If you experiment, save the result under a different name. The model and tokenizer use an immutable Hub commit rather than a moving branch.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
PROMPT = "Explain time to first token to a backend engineer in two sentences."
MAX_NEW_TOKENS = 64
SEED = 2025
RESULT_PATH = "topic-01-baseline.json"

## 1. Reproducible environment

In Colab, select **Runtime → Change runtime type → T4 GPU** before continuing. The cell uses `uv` as the installer but installs into the active notebook environment. Restart the runtime only if Colab asks you to.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.22"], check=True)
subprocess.run(["uv", "pip", "install", "--system",
                "transformers==4.51.3", "accelerate==1.6.0",
                "safetensors==0.5.3"], check=True)
print("Environment ready.")

In [ ]:
import json, os, platform, threading, time
from datetime import datetime, timezone
from importlib.metadata import version

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer, pipeline, set_seed

assert torch.cuda.is_available(), "GPU not found. In Colab, enable a GPU runtime."
set_seed(SEED)
gpu = torch.cuda.get_device_properties(0)
environment = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": version("transformers"),
    "cuda_runtime": torch.version.cuda,
    "gpu": gpu.name,
    "gpu_total_gib": round(gpu.total_memory / 2**30, 2),
}
environment

**Checkpoint:** the output must name a GPU and the pinned `transformers` version. Save this metadata; “ran on Colab” is not enough provenance.

## 2. Cold lifecycle: resolve, download and load

This timer includes cache lookup/download and model construction. Therefore it is a lifecycle measurement, not request latency. Re-running the cell measures a different (warm-cache) condition.

In [ ]:
torch.cuda.empty_cache()
load_started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    torch_dtype=torch.float16,
    device_map="cuda",
)
model.eval()
load_seconds = time.perf_counter() - load_started
weights_allocated_gib = torch.cuda.memory_allocated() / 2**30
print({
    "load_seconds": round(load_seconds, 3),
    "allocated_after_load_gib": round(weights_allocated_gib, 3),
})

## 3. Build one explicit request

The chat template is part of the artifact contract. We count **model input tokens after templating**, not characters in the user prompt.

In [ ]:
messages = [{"role": "user", "content": PROMPT}]
rendered_prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(rendered_prompt, return_tensors="pt").to(model.device)
input_tokens = int(inputs.input_ids.shape[1])
print(rendered_prompt)
print("input_tokens =", input_tokens)

### First functional baseline: `transformers.pipeline`

The pipeline is a convenient application API. We reuse the already loaded model and tokenizer so this cell does not create a second GPU copy. It proves functional generation, but it does not expose a production request queue or an API-level streaming contract.

In [ ]:
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
pipeline_result = generator(
    rendered_prompt, max_new_tokens=32, do_sample=False,
    return_full_text=False, pad_token_id=tokenizer.eos_token_id,
)
print(pipeline_result[0]["generated_text"])

## 4. Warm-up, then a non-streaming reference

Warm-up is outside the measured interval. CUDA operations are asynchronous, so synchronization is required at both timing boundaries.

In [ ]:
with torch.inference_mode():
    _ = model.generate(**inputs, max_new_tokens=4, do_sample=False, pad_token_id=tokenizer.eos_token_id)
torch.cuda.synchronize()

torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
with torch.inference_mode():
    sequence = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
torch.cuda.synchronize()
e2e_seconds = time.perf_counter() - started
generated_ids = sequence[0, input_tokens:]
output_tokens = int(generated_ids.numel())
output_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
nonstream = {
    "e2e_seconds": e2e_seconds,
    "input_tokens": input_tokens,
    "output_tokens": output_tokens,
    "output_tokens_per_second": output_tokens / e2e_seconds,
    "peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
}
print(output_text)
{k: round(v, 4) if isinstance(v, float) else v for k, v in nonstream.items()}

## 5. Client-observed streaming timeline

`TextIteratorStreamer` emits decoded text chunks, not guaranteed one-token chunks. Therefore the notebook reports **time to first non-empty chunk** and chunk gaps. It does not mislabel them as exact per-token latency. Final token count comes from generated token IDs.

In [ ]:
streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
thread_result = {}

def generate_in_thread():
    with torch.inference_mode():
        thread_result["sequence"] = model.generate(
            **inputs, streamer=streamer, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )

torch.cuda.synchronize()
stream_started = time.perf_counter()
worker = threading.Thread(target=generate_in_thread)
worker.start()
chunk_times, chunks = [], []
for chunk in streamer:
    if chunk:
        chunk_times.append(time.perf_counter() - stream_started)
        chunks.append(chunk)
worker.join()
torch.cuda.synchronize()
stream_e2e = time.perf_counter() - stream_started
stream_ids = thread_result["sequence"][0, input_tokens:]
stream_output_tokens = int(stream_ids.numel())
inter_chunk = [b - a for a, b in zip(chunk_times, chunk_times[1:])]
streaming = {
    "time_to_first_nonempty_chunk_seconds": chunk_times[0] if chunk_times else None,
    "e2e_seconds": stream_e2e,
    "nonempty_chunks": len(chunks),
    "output_tokens": stream_output_tokens,
    "mean_inter_chunk_seconds": sum(inter_chunk) / len(inter_chunk) if inter_chunk else None,
}
print("".join(chunks))
{k: round(v, 4) if isinstance(v, float) else v for k, v in streaming.items()}

### Interpret before exporting

Answer in your lab notes:

1. Which measured interval includes queue time? (There is no cross-request server queue in this notebook.)
2. Why is the first decoded chunk only a proxy for API-level TTFT?
3. Which conclusion can this single-request run support? Which production claim can it **not** support?
4. What changed between the cold lifecycle measurement and the warm request measurement?

## 6. Export audit evidence

The JSON keeps raw-enough facts to review the run. It intentionally labels proxies precisely.

In [ ]:
result = {
    "schema_version": 1,
    "experiment": "topic-01-single-request-baseline",
    "artifact": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "dtype": "float16"},
    "workload": {
        "prompt": PROMPT, "input_tokens": input_tokens,
        "max_new_tokens": MAX_NEW_TOKENS, "do_sample": False, "seed": SEED,
        "load_model": "single request, no server queue",
    },
    "environment": environment,
    "lifecycle": {"load_seconds": load_seconds, "allocated_after_load_gib": weights_allocated_gib},
    "nonstreaming": nonstream,
    "streaming_proxy": streaming,
}
with open(RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
print(f"Wrote {RESULT_PATH} ({os.path.getsize(RESULT_PATH)} bytes)")

In [ ]:
# Minimal integrity checks: fail loudly before you commit the artifact.
assert result["artifact"]["revision"] == MODEL_REVISION
assert result["nonstreaming"]["input_tokens"] > 0
assert result["nonstreaming"]["output_tokens"] > 0
assert result["streaming_proxy"]["time_to_first_nonempty_chunk_seconds"] > 0
assert result["streaming_proxy"]["e2e_seconds"] >= result["streaming_proxy"]["time_to_first_nonempty_chunk_seconds"]
print("Baseline artifact checks passed.")

## 7. Create and validate your round-01 repository

1. Create a repository from `submission-template/` in the course repository.
2. Follow `submission-template/README.md`; do not invent a different file layout.
3. Add `topic-01-baseline.json` under `evidence/` and a short `evidence/topic-01-reflection.md` answering the four questions above.
4. Run the current Colab self-check described in the template README. The judge output—not a successful notebook cell alone—is the acceptance criterion.
5. Commit only source and evidence. Never commit Hugging Face tokens, caches or model weights.
6. Create the `round-01` tag only after the self-check passes.

**Milestone:** working environment + immutable `round-01` tag that passes the self-check.

## 8. Required self-study: end-to-end boundary trace

Choose one OpenAI-compatible streaming endpoint. Draw the path from client call through gateway/router/engine/GPU and back to the final chunk. At each boundary record: owner, input/output contract, one timestamp or metric, one failure mode, and cancellation behavior. Mark clearly which statements you observed and which you inferred.